<a href="https://colab.research.google.com/github/T-Chiunzi/Tava-Smart-Finance-Assistant/blob/main/Savings_Assistant_Tester.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Pseudocode

Send greetings to the user

Display Dashboard

Ask the user to input their salary

Ask user what they want to save for and how much of their salary they want to dedicate to it

User inputs their savings goals or 0 to stop

Display to the user that they should input their expenses

Ask the user if they want to input their expenses via CSV or manually.

If via CSV, run it

If manually, ask the user to input expenses or 0 to stop

Read the data

Output table of expenses

Output a pie chart of expenses

If expenses are more than what they want to save, then advise which expenses they should reduce

Ask the user for how long they want to save for their savings goal by months or indefinitely (for emergency savings)

Ask the user if they want to increase the amount they save by (by % or by money)

Display a timeline of savings with visual of a person running along the line


In [ ]:
# Initialize a global dictionary to hold the user's financial profile
user_profile = {
    "name": "",
    "monthly_salary": 0.0,
    "emergency_goal_monthly": 0.0,
    "has_secondary_goal": False,
    "secondary_goal_amount": 0.0,
    "secondary_goal_date": "",
    "total_expenses": 0.0,
    "calculated_savings_capacity": 0.0
}

In [21]:
def process_user_profile(name, salary, emergency_amount, use_secondary, secondary_amount, secondary_date):
    global user_profile

    # Save values into our dictionary
    user_profile["name"] = name
    user_profile["monthly_salary"] = float(salary) if salary else 0.0
    user_profile["emergency_goal_monthly"] = float(emergency_amount) if emergency_amount else 0.0
    user_profile["has_secondary_goal"] = use_secondary

    if use_secondary:
        user_profile["secondary_goal_amount"] = float(secondary_amount) if secondary_amount else 0.0
        user_profile["secondary_date"] = secondary_date
    else:
        user_profile["secondary_goal_amount"] = 0.0
        user_profile["secondary_date"] = ""

    # Generate a friendly confirmation message for the UI
    confirmation_msg = f"### 🎉 Profile Saved Successfully!\n" \
                       f"Welcome **{name}**! Your monthly salary of **${user_profile['monthly_salary']:,.2f}** " \
                       f"and emergency savings goal of **${user_profile['emergency_goal_monthly']:,.2f}/month** have been recorded."

    if use_secondary:
        confirmation_msg += f"\n\nWe are also tracking your secondary goal of **${user_profile['secondary_goal_amount']:,.2f}** by **{secondary_date}**."

    return confirmation_msg

In [22]:
import pandas as pd
from datetime import datetime

def calculate_savings(manual_expenses, csv_file):
    global user_profile

    # 1. Determine total expenses (Check CSV first, then fall back to manual input)
    if csv_file is not None:
        try:
            # Read the uploaded CSV file using pandas
            # Note: We assume the CSV has an 'Amount' or 'Cost' column.
            # If your sample CSV has a specific column name, change 'Amount' to match it.
            df = pd.read_csv(csv_file.name)

            if 'Amount' in df.columns:
                total_expenses = df['Amount'].abs().sum()
            elif 'Cost' in df.columns:
                total_expenses = df['Cost'].abs().sum()
            else:
                # Fallback: if no recognized column, sum the first numeric column found
                numeric_cols = df.select_dtypes(include=['number']).columns
                if len(numeric_cols) > 0:
                    total_expenses = df[numeric_cols[0]].abs().sum()
                else:
                    return "⚠️ Error: Could not find a numeric expense column in your CSV file."
        except Exception as e:
            return f"⚠️ Error processing CSV file: {str(e)}"
    else:
        # If no CSV was uploaded, use the manually typed numeric value
        total_expenses = float(manual_expenses) if manual_expenses else 0.0

    # Save total expenses to our memory dictionary
    user_profile["total_expenses"] = total_expenses

    # 2. Financial Math Calculations
    salary = user_profile["monthly_salary"]
    leftover_income = salary - total_expenses
    user_profile["calculated_savings_capacity"] = leftover_income

    emergency_target = user_profile["emergency_goal_monthly"]
    total_needed_monthly = emergency_target

    # Logic for secondary goal calculation if it exists
    secondary_details = ""
    if user_profile["has_secondary_goal"] and user_profile["secondary_goal_amount"] > 0:
        sec_amount = user_profile["secondary_goal_amount"]
        sec_date_str = user_profile["secondary_date"]

        # Simple date math to find remaining months (Defaulting to 1 if parsing fails or date passed)
        months_remaining = 1
        try:
            target_date = datetime.strptime(sec_date_str, "%Y-%m-%d")
            current_date = datetime.now()
            # Calculate total months difference
            months_remaining = (target_date.year - current_date.year) * 12 + target_date.month - current_date.month
            if months_remaining <= 0:
                months_remaining = 1
        except:
            pass # Fallback to 1 month if date format isn't YYYY-MM-DD yet

        sec_monthly_needed = sec_amount / months_remaining
        total_needed_monthly += sec_monthly_needed
        secondary_details = f"* **Secondary Goal Monthly Target:** ${sec_monthly_needed:,.2f} (To reach ${sec_amount:,.2f} by {sec_date_str})\n"

    # 3. Generate the Financial Report Output String
    report = f"## 📊 Financial Analysis Report for {user_profile['name'] if user_profile['name'] else 'User'}\n" \
             f"---\n" \
             f"* **Total Income:** ${salary:,.2f}\n" \
             f"* **Total Logged Expenses:** ${total_expenses:,.2f}\n" \
             f"* **Remaining Cash Flow:** ${leftover_income:,.2f}\n\n" \
             f"### 🎯 Savings Commitments:\n" \
             f"* **Emergency Fund Target:** ${emergency_target:,.2f}/month\n" \
             f"{secondary_details}" \
             f"* **Total Ideal Monthly Savings:** ${total_needed_monthly:,.2f}\n" \
             f"---\n"

    if leftover_income >= total_needed_monthly:
        report += f"### ✅ Status: On Track!\n" \
                  f"Excellent job! You have enough cash flow to cover your goals and still have " \
                  f"**${leftover_income - total_needed_monthly:,.2f}** remaining for leisurely spending."
    elif leftover_income >= emergency_target:
        report += f"### ⚠️ Status: Partial Progress\n" \
                  f"You can comfortably cover your Emergency Fund, but you are short by " \
                  f"**${total_needed_monthly - leftover_income:,.2f}** to hit your secondary goal target simultaneously. Consider reducing non-essential spending."
    else:
        report += f"### ❌ Status: Budget Deficit\n" \
                  f"Warning! Your current monthly expenses do not leave you enough room to hit your baseline savings goals. " \
                  f"You are short by **${total_needed_monthly - leftover_income:,.2f}** this month. Ask your Chat Coach for budgeting tips!"

    return report

def reset_expense_tracker():
    global user_profile
    # Reset internal memory values
    user_profile["total_expenses"] = 0.0
    user_profile["calculated_savings_capacity"] = 0.0

    # Return values to clear the Gradio elements (Manual textbox, CSV upload, Markdown status text)
    return None, None, "🔄 Expenses cleared! You can now input a new manual amount or upload a new CSV file."

In [23]:
# --- STEP 6 SIMULATION: TESTING THE ENGINE ---

# 1. Simulate the user saving their profile data (Tab 1 action)
print("--- Simulating Tab 1: Profile Setup ---")
setup_result = process_user_profile(
    name="Alex",
    salary=5000,
    emergency_amount=500,
    use_secondary=True,
    secondary_amount=2400,
    secondary_date="2027-05-25" # Exactly 12 months from now
)
print(setup_result)
print("\n" + "="*50 + "\n")

# 2. Simulate the user calculating manual expenses (Tab 2 action)
print("--- Simulating Tab 2: Expense Calculation ---")
report_output = calculate_savings(manual_expenses=2800, csv_file=None)
print(report_output)

--- Simulating Tab 1: Profile Setup ---
### 🎉 Profile Saved Successfully!
Welcome **Alex**! Your monthly salary of **$5,000.00** and emergency savings goal of **$500.00/month** have been recorded.

We are also tracking your secondary goal of **$2,400.00** by **2027-05-25**.


--- Simulating Tab 2: Expense Calculation ---
## 📊 Financial Analysis Report for Alex
---
* **Total Income:** $5,000.00
* **Total Logged Expenses:** $2,800.00
* **Remaining Cash Flow:** $2,200.00

### 🎯 Savings Commitments:
* **Emergency Fund Target:** $500.00/month
* **Secondary Goal Monthly Target:** $200.00 (To reach $2,400.00 by 2027-05-25)
* **Total Ideal Monthly Savings:** $700.00
---
### ✅ Status: On Track!
Excellent job! You have enough cash flow to cover your goals and still have **$1,500.00** remaining for leisurely spending.
